# FreightLake — PostgreSQL EDA

Exploratory analysis of the structured logistics source tables before Bronze ingestion.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, inspect, text
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5432')}"
    f"/{os.getenv('POSTGRES_DB', 'freight_lake')}"
)

print('Connected to PostgreSQL')

## 1. Tables

In [ ]:
inspector = inspect(engine)
tables = inspector.get_table_names(schema='public')
tables

## 2. Row Counts

In [ ]:
counts = []
for table in tables:
    n = pd.read_sql(text(f'SELECT COUNT(*) AS row_count FROM public."{table}"'), engine).iloc[0, 0]
    counts.append({'table': table, 'row_count': n})

pd.DataFrame(counts).sort_values('row_count', ascending=False)

## 3. Schema and Data Types

In [ ]:
for table in tables:
    print(f'\n### {table}')
    for col in inspector.get_columns(table, schema='public'):
        print(f"{col['name']}: {col['type']}")

## 4. Null Analysis

In [ ]:
for table in tables:
    df = pd.read_sql(f'SELECT * FROM public."{table}"', engine)
    nulls = df.isna().sum().to_frame('null_count')
    nulls['null_pct'] = (nulls['null_count'] / len(df) * 100).round(2) if len(df) else 0
    print(f'\n### {table}')
    display(nulls[nulls['null_count'] > 0].sort_values('null_count', ascending=False))

## 5. Duplicate Analysis

In [ ]:
for table in tables:
    df = pd.read_sql(f'SELECT * FROM public."{table}"', engine)
    print(f'{table}: {df.duplicated().sum()} duplicate rows')

## 6. `updated_at` Analysis

In [ ]:
for table in tables:
    if 'updated_at' in [c['name'] for c in inspector.get_columns(table, schema='public')]:
        df = pd.read_sql(f'SELECT MIN(updated_at) AS min_updated_at, MAX(updated_at) AS max_updated_at FROM public."{table}"', engine)
        print(f'\n{table}')
        display(df)